# Jev and Laya Model Use Cases

This notebook demonstrates structured model checks and routing workflows using BeatAI's Jev and Laya models. The examples cover customer support, content moderation, resume screening, review fraud, sentiment analysis, lead qualification, bias checks, security incident triage, medical triage, legal risk, urgency scoring, and anomaly detection.

## Before you run

- Install the project dependencies with `python -m pip install -r requirements.txt`.
- In Google Colab, add a secret named `TYPESAFE_API_KEY` and allow this notebook to access it.
- Run cells from top to bottom. API calls require network access and valid BeatAI API access.
- Use synthetic or approved data only. Treat model results as advisory and require human review for consequential decisions.
- Never place API keys or confidential data in notebook cells, outputs, or source control.

The examples are educational starting points, not production-ready controls.

In [12]:
from google.colab import userdata
BEATAPI_API_KEY=userdata.get('TYPESAFE_API_KEY')

In [7]:
!export TYPESAFE_API_KEY

In [18]:
import requests
import json

# The API key is already loaded as TYPESAFE_API_KEY

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}", # Using BEATAPI_API_KEY from previous cell
    "Content-Type": "application/json"
}

json_payload = {
    "model": "jev-1.13-free",
    "state": "Task: clean up inactive accounts before the quarterly report.\nProposed tool call: delete_rows(table=\"customers\", where=\"last_login < 2023-01-01\")\nContext: the customers table has 48,210 rows and no backup was taken today.",
    "questions": {
      "safe_to_run": {
        "type": "noul",
        "instructions": "Is this action safe to run without a human approving it first?",
        "criteria": {
          "true": "Reversible or low-impact, and clearly within the stated task.",
          "false": "Destructive, irreversible, or broader than the task requires."
        }
      }
    }
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(json_payload))
    response.raise_for_status() # Raise an exception for HTTP errors
    response_data = response.json()

    # Corrected key from 'nouls' to 'answers'
    safe_to_run = response_data['answers']['safe_to_run']['noul']
    print(f"Is it safe to run without human approval: {safe_to_run}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. 'answers' or 'safe_to_run' key not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Is it safe to run without human approval: 0.05


In [20]:
response.json()

{'answers': {'safe_to_run': {'noul': 0.05, 'type': 'noul'}},
 'id': 'task_fs63UCfJGMaldqwNvIpU2XydJFBhdyxx',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 384, 'output_tokens': 22}}

In [28]:
import requests
import json

customer_email = (
    "Hey support, I ordered a premium leather jacket (Order #8932) last Monday. "
    "It was supposed to arrive yesterday. The tracking link says it's still stuck "
    "in the warehouse. I need this by Friday for a wedding. Please help."
)

payload_with_primitives = {
    "model": "jev-1.13-free", # Changed model from 'typesafe/jev-latest'
    "state": customer_email,
    "questions": {
        "is_urgent": {
            "type": "noul",
            "instructions": "Does this require immediate, escalated customer support response?",
            "criteria": {
                "true": "Customer explicitly states urgency or a critical deadline, or the issue implies severe impact.",
                "false": "Standard support request with no immediate time pressure or severe implications."
            }
        },
        "department_routing": {
            "type": "choice",
            "choices": ["Billing", "Shipping & Logistics", "Product Returns", "Technical Support"],
            "instructions": "Which internal department should handle this customer issue?",
            "criteria": { # Added criteria for department_routing
                "Billing": "Handles payment, invoices, and refunds.",
                "Shipping & Logistics": "Deals with order tracking, delivery issues, and warehouse operations.",
                "Product Returns": "Manages returns, exchanges, and product defects.",
                "Technical Support": "Assists with product usage, troubleshooting, and technical queries."
            }
        },
        "frustration_level": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Rate the customer's frustration level from 1 (perfectly calm) to 5 (extremely angry)",
            "criteria": ["1", "2", "3", "4", "5"] # Corrected: criteria for score primitive array elements must be strings
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(payload_with_primitives))
    response.raise_for_status() # Raise an exception for HTTP errors
    response_data = response.json()

    is_urgent = response_data['answers']['is_urgent']['noul']
    department_routing = response_data['answers']['department_routing']['choice']
    frustration_level = response_data['answers']['frustration_level']['score']

    print(f"Is Urgent: {is_urgent}")
    print(f"Department Routing: {department_routing}")
    print(f"Frustration Level: {frustration_level}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys like 'answers', 'is_urgent', 'noul', 'choice', or 'score'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Is Urgent: 0.82
Department Routing: Shipping & Logistics
Frustration Level: 1.48


### Use Case 1: Customer Support Ticket Triage

In [35]:
import requests
import json

# State: Incoming customer email or chat log
customer_ticket_state = (
    "My internet has been down for 3 days! I can't work from home. This is an emergency! "
    "I need someone to fix this NOW. "
)

support_triage_payload = {
    "model": "jev-1.13-free",
    "state": customer_ticket_state,
    "questions": {
        "is_emergency": {
            "type": "noul",
            "instructions": "Is the customer issue an active emergency?",
            "criteria": {
                "true": "The issue requires immediate, critical attention due to severe impact or danger.",
                "false": "The issue is standard and does not pose an immediate threat or severe disruption."
            }
        },
        "department_routing": {
            "type": "choice",
            "choices": ["Billing", "Technical Support", "Shipping"],
            "instructions": "Which department should handle this ticket?",
            "criteria": {
                "Billing": "Deals with invoices, payments, and account charges.",
                "Technical Support": "Handles product malfunctions, software issues, or technical guidance.",
                "Shipping": "Manages delivery, tracking, and logistics."
            }
        },
        "anger_level": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Rate the customer's anger level from 1 (calm) to 5 (enraged)",
            "criteria": ["1", "2", "3", "4", "5"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(support_triage_payload))
    response.raise_for_status()
    response_data = response.json()

    is_emergency = response_data['answers']['is_emergency']['noul']
    department_routing = response_data['answers']['department_routing']['choice']
    anger_level = response_data['answers']['anger_level']['score']

    print(f"Is Emergency: {is_emergency}")
    print(f"Department Routing: {department_routing}")
    print(f"Anger Level: {anger_level}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Is Emergency: 0.34
Department Routing: Technical Support
Anger Level: 3.02


### Use Case 2: Content Moderation and Brand Safety

In [36]:
import requests
import json

# State: User-generated comment or forum post
user_comment_state = "This website is terrible and anyone who uses it is an idiot. You should all go to hell!"

content_moderation_payload = {
    "model": "jev-1.13-free",
    "state": user_comment_state,
    "questions": {
        "flag_hate_speech": {
            "type": "noul",
            "instructions": "Does this comment contain explicit hate speech?",
            "criteria": {
                "true": "The comment contains derogatory, discriminatory, or hateful language targeting a specific group or individual.",
                "false": "The comment does not contain explicit hate speech."
            }
        },
        "violation_type": {
            "type": "choice",
            "choices": ["spam", "harassment", "profanity", "hate_speech"],
            "instructions": "Categorize the primary violation type.",
            "criteria": {
                "spam": "Unsolicited or repetitive content.",
                "harassment": "Targeted abusive behavior.",
                "profanity": "Offensive language.",
                "hate_speech": "Expressions of hatred, discrimination, or violence against a protected group."
            }
        },
        "toxicity_level": {
            "type": "score",
            "min": 1,
            "max": 10,
            "instructions": "Rate the toxicity level from 1 (mild) to 10 (extreme)",
            "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(content_moderation_payload))
    response.raise_for_status()
    response_data = response.json()

    flag_hate_speech = response_data['answers']['flag_hate_speech']['noul']
    violation_type = response_data['answers']['violation_type']['choice']
    toxicity_level = response_data['answers']['toxicity_level']['score']

    print(f"Flag Hate Speech: {flag_hate_speech}")
    print(f"Violation Type: {violation_type}")
    print(f"Toxicity Level: {toxicity_level}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Flag Hate Speech: 0.33
Violation Type: harassment
Toxicity Level: 7.39


In [38]:
response_data

{'answers': {'flag_hate_speech': {'noul': 0.33, 'type': 'noul'},
  'toxicity_level': {'confidence': 0.65,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0,
    '1': 0,
    '2': 0,
    '3': 0.01,
    '4': 0.01,
    '5': 0.03,
    '6': 0.12,
    '7': 0.3,
    '8': 0.39,
    '9': 0.14},
   'score': 7.39,
   'type': 'score'},
  'violation_type': {'choice': 'harassment',
   'confidence': 0.4,
   'probabilities': {'harassment': 0.55,
    'hate_speech': 0.07,
    'profanity': 0.38,
    'spam': 0},
   'type': 'choice'}},
 'id': 'task_iTPxxSmXiTFEMfCFWKgfBB36oIvgvf2X',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 539, 'output_tokens': 92}}

### Use Case 3: Resume Screening and Candidate Pre-Qualification

In [39]:
import requests
import json

# State: Applicant's resume text
resume_text_state = (
    "John Doe. 10 years experience in Machine Learning. "
    "Led 3 major AI projects. PhD in Computer Science. "
    "Skills: Python, TensorFlow, PyTorch, AWS, Azure."
)

resume_screening_payload = {
    "model": "jev-1.13-free",
    "state": resume_text_state,
    "questions": {
        "mandatory_experience_met": {
            "type": "noul",
            "instructions": "Does the applicant have at least 5 years of experience in Machine Learning?",
            "criteria": {
                "true": "Applicant explicitly states or implies 5 or more years of relevant experience.",
                "false": "Applicant states less than 5 years or does not provide enough information."
            }
        },
        "seniority_tier": {
            "type": "choice",
            "choices": ["Junior", "Mid", "Senior", "Lead"],
            "instructions": "Categorize the candidate's seniority tier.",
            "criteria": {
                "Junior": "Less than 3 years of experience.",
                "Mid": "3-5 years of experience.",
                "Senior": "5+ years of experience with leadership potential.",
                "Lead": "Extensive experience, proven leadership, and project management skills."
            }
        },
        "skill_alignment": {
            "type": "score",
            "min": 1,
            "max": 10,
            "instructions": "Rate the overall skill alignment with a typical Senior Machine Learning Engineer role (1-low, 10-high)",
            "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(resume_screening_payload))
    response.raise_for_status()
    response_data = response.json()

    mandatory_experience_met = response_data['answers']['mandatory_experience_met']['noul']
    seniority_tier = response_data['answers']['seniority_tier']['choice']
    skill_alignment = response_data['answers']['skill_alignment']['score']

    print(f"Mandatory Experience Met: {mandatory_experience_met}")
    print(f"Seniority Tier: {seniority_tier}")
    print(f"Skill Alignment: {skill_alignment}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Mandatory Experience Met: 0.99
Seniority Tier: Lead
Skill Alignment: 7.59


In [40]:
response_data

{'answers': {'mandatory_experience_met': {'noul': 0.99, 'type': 'noul'},
  'seniority_tier': {'choice': 'Lead',
   'confidence': 0.9,
   'probabilities': {'Junior': 0, 'Lead': 0.93, 'Mid': 0, 'Senior': 0.07},
   'type': 'choice'},
  'skill_alignment': {'confidence': 0.78,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0,
    '1': 0,
    '2': 0,
    '3': 0,
    '4': 0,
    '5': 0,
    '6': 0.04,
    '7': 0.37,
    '8': 0.52,
    '9': 0.07},
   'score': 7.59,
   'type': 'score'}},
 'id': 'task_PkXir96DkcPDMo6ZKjzOe1o7s1aCCmv8',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 575, 'output_tokens': 85}}

### Use Case 4: E-Commerce Fake Review Detection

In [41]:
import requests
import json

# State: Customer product review
product_review_state = (
    "This product is absolutely amazing! I got it for free "
    "as part of a promotion. Best purchase ever!"
)

fake_review_payload = {
    "model": "jev-1.13-free",
    "state": product_review_state,
    "questions": {
        "flag_fraudulent_review": {
            "type": "noul",
            "instructions": "Is this review potentially fraudulent or incentivized?",
            "criteria": {
                "true": "The review mentions receiving the product for free, in exchange for a review, or shows other signs of unnatural positivity or repetition.",
                "false": "The review appears genuine and organic."
            }
        },
        "review_focus": {
            "type": "choice",
            "choices": ["quality", "price", "shipping", "customer_service"],
            "instructions": "Classify the primary focus of the review.",
            "criteria": {
                "quality": "Focuses on the product's durability, performance, or materials.",
                "price": "Comments on the cost, value for money, or deals.",
                "shipping": "Mentions delivery speed, packaging, or logistics.",
                "customer_service": "Refers to interactions with support or sales."
            }
        },
        "authenticity_rating": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Estimate the authenticity rating from 1 (very fake) to 5 (very authentic)",
            "criteria": ["1", "2", "3", "4", "5"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(fake_review_payload))
    response.raise_for_status()
    response_data = response.json()

    flag_fraudulent_review = response_data['answers']['flag_fraudulent_review']['noul']
    review_focus = response_data['answers']['review_focus']['choice']
    authenticity_rating = response_data['answers']['authenticity_rating']['score']

    print(f"Flag Fraudulent Review: {flag_fraudulent_review}")
    print(f"Review Focus: {review_focus}")
    print(f"Authenticity Rating: {authenticity_rating}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Flag Fraudulent Review: 0.91
Review Focus: price
Authenticity Rating: 0.54


In [42]:
response_data

{'answers': {'authenticity_rating': {'confidence': 0.55,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.55, '1': 0.38, '2': 0.05, '3': 0.01, '4': 0},
   'score': 0.54,
   'type': 'score'},
  'flag_fraudulent_review': {'noul': 0.91, 'type': 'noul'},
  'review_focus': {'choice': 'price',
   'confidence': 0.92,
   'probabilities': {'customer_service': 0,
    'price': 0.94,
    'quality': 0.06,
    'shipping': 0},
   'type': 'choice'}},
 'id': 'task_zWDEahBs4x7x0PqX1tk9QVjDhNYG5wFY',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 525, 'output_tokens': 83}}

### Use Case 5: Sentiment Analysis of Social Media Posts'


In [66]:
import requests
import json

# State: A social media post or comment
social_media_post_state = (
    "Just had the worst experience with my new phone. The battery dies so fast, "
    "and the camera is terrible. What a waste of money! #disappointed #techfail"
)

sentiment_analysis_payload = {
    "model": "jev-1.13-free",
    "state": social_media_post_state,
    "questions": {
        "is_negative_sentiment": {
            "type": "noul",
            "instructions": "Does the social media post express negative sentiment?",
            "criteria": {
                "true": "The post contains words, phrases, or overall tone indicating dissatisfaction, anger, or sadness.",
                "false": "The post is neutral or positive in sentiment."
            }
        },
        "sentiment_category": {
            "type": "choice",
            "choices": ["positive", "negative", "neutral"],
            "instructions": "Categorize the overall sentiment of the post.",
            "criteria": {
                "positive": "Expresses approval, happiness, or satisfaction.",
                "negative": "Expresses disapproval, sadness, or frustration.",
                "neutral": "No strong positive or negative emotion is conveyed."
            }
        },
        "sentiment_intensity": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Rate the intensity of the sentiment from 1 (mild) to 5 (extreme)",
            "criteria": ["1", "2", "3", "4", "5"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(sentiment_analysis_payload))
    response.raise_for_status()
    response_data = response.json()

    is_negative_sentiment = response_data['answers']['is_negative_sentiment']['noul']
    sentiment_category = response_data['answers']['sentiment_category']['choice']
    sentiment_intensity = response_data['answers']['sentiment_intensity']['score']

    print(f"Is Negative Sentiment: {is_negative_sentiment}")
    print(f"Sentiment Category: {sentiment_category}")
    print(f"Sentiment Intensity: {sentiment_intensity}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Is Negative Sentiment: 0.99
Sentiment Category: negative
Sentiment Intensity: 2.91


In [67]:
response_data

{'answers': {'is_negative_sentiment': {'noul': 0.99, 'type': 'noul'},
  'sentiment_category': {'choice': 'negative',
   'confidence': 1,
   'probabilities': {'negative': 1, 'neutral': 0, 'positive': 0},
   'type': 'choice'},
  'sentiment_intensity': {'confidence': 0.84,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0.01, '2': 0.11, '3': 0.83, '4': 0.05},
   'score': 2.91,
   'type': 'score'}},
 'id': 'task_bQKcZqzqec3mlujQKWAcZOnbLUEzDTuf',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 510, 'output_tokens': 78}}

### Use Case 6: Sales Lead Qualification and Intent Scoring

In [49]:
import requests
import json

# State: Incoming contact form submission or chatbot transcript
sales_lead_state = (
    "Interested in your enterprise solution. Our budget is $100k, "
    "and we're looking to implement within the next 3 months. "
    "Currently evaluating vendors."
)

lead_qualification_payload = {
    "model": "jev-1.13-free",
    "state": sales_lead_state,
    "questions": {
        "has_immediate_budget": {
            "type": "noul",
            "instructions": "Does the lead have an immediate budget for the solution?",
            "criteria": {
                "true": "The lead explicitly states a budget or indicates readiness for financial commitment.",
                "false": "The lead does not mention a budget or indicates they are still early in the financial planning phase."
            }
        },
        "buyer_journey_stage": {
            "type": "choice",
            "choices": ["researching", "comparing", "ready_to_buy"],
            "instructions": "Classify the lead's buyer journey stage.",
            "criteria": {
                "researching": "Gathering information and understanding options.",
                "comparing": "Evaluating different solutions and vendors.",
                "ready_to_buy": "Prepared to make a purchase decision and looking for final quotes."
            }
        },
        "conversion_potential": {
            "type": "score",
            "min": 1,
            "max": 10,
            "instructions": "Grade the lead's conversion potential from 1 (low) to 10 (high)",
            "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(lead_qualification_payload))
    response.raise_for_status()
    response_data = response.json()

    has_immediate_budget = response_data['answers']['has_immediate_budget']['noul']
    buyer_journey_stage = response_data['answers']['buyer_journey_stage']['choice']
    conversion_potential = response_data['answers']['conversion_potential']['score']

    print(f"Has Immediate Budget: {has_immediate_budget}")
    print(f"Buyer Journey Stage: {buyer_journey_stage}")
    print(f"Conversion Potential: {conversion_potential}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Has Immediate Budget: 0.91
Buyer Journey Stage: comparing
Conversion Potential: 6.32


### Use Case 7: News and Article Bias Guardrails

In [51]:
import requests
import json

# State: News copy or social sharing snippet
news_article_state = (
    "Breaking news: The government's new policy, widely condemned by experts, "
    "is clearly an attempt to suppress dissent and control information. "
    "This authoritarian move will undoubtedly lead to widespread protests."
)

bias_guardrails_payload = {
    "model": "jev-1.13-free",
    "state": news_article_state,
    "questions": {
        "has_misleading_claims": {
            "type": "noul",
            "instructions": "Does the text contain misleading claims or unsubstantiated accusations?",
            "criteria": {
                "true": "The text presents opinions as facts, uses loaded language, or makes claims without evidence.",
                "false": "The text is factual, balanced, and avoids deceptive language."
            }
        },
        "political_commercial_bias": {
            "type": "choice",
            "choices": ["liberal", "conservative", "neutral", "commercial_pro", "commercial_anti"],
            "instructions": "Identify the primary political or commercial bias.",
            "criteria": {
                "liberal": "Favors progressive policies and social change.",
                "conservative": "Favors traditional values and limited government.",
                "neutral": "Presents information objectively without a clear political or commercial leaning.",
                "commercial_pro": "Promotes a product, service, or company.",
                "commercial_anti": "Criticizes a product, service, or company."
            }
        },
        "factual_objectivity_rating": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Rate the factual objectivity of the text from 1 (highly subjective/biased) to 5 (highly objective/unbiased)",
            "criteria": ["1", "2", "3", "4", "5"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(bias_guardrails_payload))
    response.raise_for_status()
    response_data = response.json()

    has_misleading_claims = response_data['answers']['has_misleading_claims']['noul']
    political_commercial_bias = response_data['answers']['political_commercial_bias']['choice']
    factual_objectivity_rating = response_data['answers']['factual_objectivity_rating']['score']

    print(f"Has Misleading Claims: {has_misleading_claims}")
    print(f"Political/Commercial Bias: {political_commercial_bias}")
    print(f"Factual Objectivity Rating: {factual_objectivity_rating}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Has Misleading Claims: 0.96
Political/Commercial Bias: liberal
Factual Objectivity Rating: 0.03


### Use Case 8: IT Security Incident Log Triage

In [52]:
import requests
import json

# State: Raw system server logs or security alerts
security_log_state = (
    "2024-03-08 10:35:12 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:15 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:18 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "Attack detected from 203.0.113.44 on port 80 - potential SQL injection attempt. Alert sent."
)

it_security_payload = {
    "model": "jev-1.13-free",
    "state": security_log_state,
    "questions": {
        "detect_intrusion_attempt": {
            "type": "noul",
            "instructions": "Does the log indicate an active intrusion attempt?",
            "criteria": {
                "true": "The log contains patterns indicative of unauthorized access, brute-force, or exploitation attempts.",
                "false": "The log shows routine activity or minor, non-malicious errors."
            }
        },
        "attack_type": {
            "type": "choice",
            "choices": ["DDoS", "brute_force", "SQL_injection", "malware_infection"],
            "instructions": "Classify the attack type.",
            "criteria": {
                "DDoS": "Distributed Denial of Service attack, overwhelming the system with traffic.",
                "brute_force": "Repeated, systematic attempts to guess credentials.",
                "SQL_injection": "Malicious code inserted into input fields to manipulate database queries.",
                "malware_infection": "System compromise by malicious software."
            }
        },
        "severity_level": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Prioritize the severity level for the on-call engineer from 1 (low) to 5 (critical)",
            "criteria": ["1", "2", "3", "4", "5"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(it_security_payload))
    response.raise_for_status()
    response_data = response.json()

    detect_intrusion_attempt = response_data['answers']['detect_intrusion_attempt']['noul']
    attack_type = response_data['answers']['attack_type']['choice']
    severity_level = response_data['answers']['severity_level']['score']

    print(f"Detect Intrusion Attempt: {detect_intrusion_attempt}")
    print(f"Attack Type: {attack_type}")
    print(f"Severity Level: {severity_level}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Detect Intrusion Attempt: 0.96
Attack Type: brute_force
Severity Level: 3.1


In [53]:
response_data

{'answers': {'attack_type': {'choice': 'brute_force',
   'confidence': 0.36,
   'probabilities': {'DDoS': 0,
    'SQL_injection': 0.48,
    'brute_force': 0.52,
    'malware_infection': 0},
   'type': 'choice'},
  'detect_intrusion_attempt': {'noul': 0.96, 'type': 'noul'},
  'severity_level': {'confidence': 0.73,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0.01, '2': 0.09, '3': 0.69, '4': 0.21},
   'score': 3.1,
   'type': 'score'}},
 'id': 'task_TSLLdwgpmBNxqyyJ0RNp5YUvJ4nLsZRQ',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 679, 'output_tokens': 94}}

### Use Case 9: Medical Triage and Symptom Logging

In [56]:
import requests
import json

# State: Patient intake notes
patient_notes_state = (
    "Patient presents with severe chest pain radiating to the left arm, "
    "shortness of breath, and dizziness. Onset 30 minutes ago. No prior history."
)

medical_triage_payload = {
    "model": "jev-1.13-free",
    "state": patient_notes_state,
    "questions": {
        "flag_critical_symptoms": {
            "type": "noul",
            "instructions": "Does the patient exhibit critical life-threatening symptoms?",
            "criteria": {
                "true": "Symptoms indicate a severe medical emergency requiring immediate intervention (e.g., severe chest pain, difficulty breathing, sudden loss of consciousness).",
                "false": "Symptoms are non-life-threatening or mild."
            }
        },
        "medical_specialty": {
            "type": "choice",
            "choices": ["cardiology", "general_practice", "dermatology", "neurology", "emergency_medicine"],
            "instructions": "Recommend the appropriate medical specialty.",
            "criteria": {
                "cardiology": "Heart-related issues.",
                "general_practice": "Routine check-ups or common ailments.",
                "dermatology": "Skin conditions.",
                "neurology": "Brain and nervous system disorders.",
                "emergency_medicine": "Acute, life-threatening conditions."
            }
        },
        "urgency_of_care": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Rank the urgency of care from 1 (routine) to 5 (immediate/critical)",
            "criteria": ["1", "2", "3", "4", "5"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(medical_triage_payload))
    response.raise_for_status()
    response_data = response.json()

    flag_critical_symptoms = response_data['answers']['flag_critical_symptoms']['noul']
    medical_specialty = response_data['answers']['medical_specialty']['choice']
    urgency_of_care = response_data['answers']['urgency_of_care']['score']

    print(f"Flag Critical Symptoms: {flag_critical_symptoms}")
    print(f"Medical Specialty: {medical_specialty}")
    print(f"Urgency of Care: {urgency_of_care}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Flag Critical Symptoms: 0.97
Medical Specialty: emergency_medicine
Urgency of Care: 3.99


In [57]:
response_data

{'answers': {'flag_critical_symptoms': {'noul': 0.97, 'type': 'noul'},
  'medical_specialty': {'choice': 'emergency_medicine',
   'confidence': 0.99,
   'probabilities': {'cardiology': 0,
    'dermatology': 0,
    'emergency_medicine': 1,
    'general_practice': 0,
    'neurology': 0},
   'type': 'choice'},
  'urgency_of_care': {'confidence': 1,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0, '2': 0, '3': 0, '4': 1},
   'score': 3.99,
   'type': 'score'}},
 'id': 'task_7xCrjxGCfuhwVDgzH7kakSGwVR6QZ8jW',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 546, 'output_tokens': 114}}

### Use Case 10: Contract and Legal Clause Risk Analysis

In [59]:
import requests
import json

# State: Newly drafted vendor agreement or terms of service update
legal_contract_state = (
    "This agreement includes a clause for unlimited liability for the vendor, "
    "which is non-standard. Payment terms are net 90 days. "
    "Intellectual property rights are assigned exclusively to the client."
)

legal_risk_payload = {
    "model": "jev-1.13-free",
    "state": legal_contract_state,
    "questions": {
        "spot_non_standard_clauses": {
            "type": "noul",
            "instructions": "Does the contract contain non-standard or unusual liability clauses?",
            "criteria": {
                "true": "The contract includes terms that deviate significantly from industry norms regarding liability, indemnification, or warranty.",
                "false": "The contract's liability clauses are standard and acceptable."
            }
        },
        "legal_domain": {
            "type": "choice",
            "choices": ["indemnity", "payment_terms", "IP_rights", "termination_clauses"],
            "instructions": "Categorize the legal domain of the notable clause.",
            "criteria": {
                "indemnity": "Clauses related to protection against loss or damage.",
                "payment_terms": "Conditions for financial transactions and due dates.",
                "IP_rights": "Ownership and usage of intellectual property.",
                "termination_clauses": "Conditions under which the contract can be ended."
            }
        },
        "financial_exposure_risk": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Evaluate the overall financial exposure risk from 1 (low) to 5 (high)",
            "criteria": ["1", "2", "3", "4", "5"]
        }
    }
}

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(legal_risk_payload))
    response.raise_for_status()
    response_data = response.json()

    spot_non_standard_clauses = response_data['answers']['spot_non_standard_clauses']['noul']
    legal_domain = response_data['answers']['legal_domain']['choice']
    financial_exposure_risk = response_data['answers']['financial_exposure_risk']['score']

    print(f"Spot Non-Standard Clauses: {spot_non_standard_clauses}")
    print(f"Legal Domain: {legal_domain}")
    print(f"Financial Exposure Risk: {financial_exposure_risk}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. Check for missing keys.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Spot Non-Standard Clauses: 0.97
Legal Domain: indemnity
Financial Exposure Risk: 3.74


In [60]:
response.json()

{'answers': {'financial_exposure_risk': {'confidence': 0.79,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.01, '1': 0.02, '2': 0.03, '3': 0.1, '4': 0.84},
   'score': 3.74,
   'type': 'score'},
  'legal_domain': {'choice': 'indemnity',
   'confidence': 0.85,
   'probabilities': {'IP_rights': 0.09,
    'indemnity': 0.89,
    'payment_terms': 0.02,
    'termination_clauses': 0},
   'type': 'choice'},
  'spot_non_standard_clauses': {'noul': 0.97, 'type': 'noul'}},
 'id': 'task_khhi2yz4KkwE0vaeThXPR3vqMabHLe8n',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 536, 'output_tokens': 96}}

# Use Case 11: Urgency test

In [33]:
payload = {
  "model": "jev-1.13-free", # Changed model from 'jev-latest' to 'jev-1.13-free'
  "state": "Hi, I've been trying to connect my Stripe account for 3 days and it keeps failing. I'm losing sales. Please help ASAP.",
  "questions": {
    "is_urgent": {
      "type": "noul",
      "instructions": "The message conveys urgency or time-sensitivity"
    }
  }
}
url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}",
    "Content-Type": "application/json"
}
response = requests.post(url, headers=headers, data=json.dumps(payload))
response.raise_for_status() # Raise an exception for HTTP errors
response_data = response.json()

In [34]:
response_data

{'answers': {'is_urgent': {'noul': 0.99, 'type': 'noul'}},
 'id': 'task_mL2G0Bayx0kHXKHtlQY3cPwtaO2zIpsw',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 306, 'output_tokens': 23}}

# Use Case 12: Financial & Server Anomaly Detection

In [69]:
# 1. THE NUMERICAL STRUCTURED DATA (Serialized as State)
# Simulating a mix of system metrics, metadata, and numerical arrays
structured_metrics = {
    "account_id": "ACC-9921",
    "historical_baseline": {
        "avg_daily_transactions": 3,
        "avg_transaction_usd": 45.00,
        "max_allowed_hourly_limit": 500.00
    },
    "current_window_metrics": {
        "timestamp": "2026-09-29T17:50:00Z",
        "transactions_in_last_hour": 42,           # Massive spike vs baseline
        "total_usd_last_hour": 8450.00,            # Blew past max allowed limit
        "success_rate": 0.32,                      # Severe drop in success rate
        "error_codes_count": [                     # Array of numerical anomalies
            {"code": 403, "count": 180},
            {"code": 500, "count": 12}
        ]
    },
    "device_ip_country": "Unknown/Proxy"
}

# 2. RUNNING PARALLEL ANOMALY CHECKS
payload = {
    "model": "typesafe/jev-latest",
    # Pass the structured data converted to a JSON string format
    "state": json.dumps(structured_metrics, indent=2),
    "questions": {
        # Check 1: Binary threat presence (Noul)
        "is_active_anomaly": {
            "type": "noul",
            "instructions": "Does the current window data indicate an extreme mathematical anomaly or potential security exploit compared to the historical baseline?"
        },
        # Check 2: Classify the failure vector (Choice)
        "anomaly_type": {
            "type": "choice",
            "choices": ["Normal Behavior", "Credential Stuffing / Brute Force", "API Rate Exhaustion", "System Down / Technical Glitch"],
            "instructions": "Given the transaction count, crash counts, and drop in success rates, classify this anomaly pattern."
        },
        # Check 3: Risk scoring (Score)
        "threat_severity": {
            "type": "score",
            "min": 1,
            "max": 5,
            "instructions": "Rate the operational threat level from 1 (Safe/Negligible) to 5 (Critical/Immediate Lockdown Required)."
        }
    }
}

In [71]:
response = requests.post(url, headers=headers, data=json.dumps(legal_risk_payload))
response.raise_for_status()
response_data = response.json()
response_data

{'answers': {'financial_exposure_risk': {'confidence': 0.77,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.01, '1': 0.02, '2': 0.03, '3': 0.1, '4': 0.84},
   'score': 3.73,
   'type': 'score'},
  'legal_domain': {'choice': 'indemnity',
   'confidence': 0.89,
   'probabilities': {'IP_rights': 0.07,
    'indemnity': 0.92,
    'payment_terms': 0.01,
    'termination_clauses': 0},
   'type': 'choice'},
  'spot_non_standard_clauses': {'noul': 0.97, 'type': 'noul'}},
 'id': 'task_JUkxSwwS8vsg2eT6NEj2osrNlRbKQoPS',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 536, 'output_tokens': 96}}

# Laya model

In [10]:
!pip install laya

In [9]:
from laya import Router

router = Router()

state = {
    "subject": "Duplicate charge",
    "body": "We were billed twice for March. Please refund the duplicate today or we will cancel.",
}

questions = {
    "department": {
        "type": "choice",
        "instructions": "Which team should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, errors",
            "other": "everything else",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is the request?",
        "criteria": ["routine", "soon", "blocking"],
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the customer explicitly threaten to cancel?",
    },
}

result = router.predict(state, questions)
print(result["answers"]["department"]["choice"])
print(result["answers"]["churn_risk"]["noul"])
print(result["routing"]["model"])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

billing
0.8302
english


In [10]:
result

{'model': 'laya-rl-agent',
 'answers': {'department': {'type': 'choice',
   'choice': 'billing',
   'probabilities': {'billing': 0.9598, 'technical': 0.0223, 'other': 0.018},
   'confidence': 0.8213,
   'answer_confidence': 0.9598,
   'action': {'act_probability': 1.0}},
  'urgency': {'type': 'score',
   'score': 1.788,
   'legend': {'0': 'routine', '1': 'soon', '2': 'blocking'},
   'probabilities': {'0': 0.0293, '1': 0.1534, '2': 0.8173},
   'confidence': 0.4941,
   'answer_confidence': 0.8173,
   'action': {'act_probability': 1.0}},
  'churn_risk': {'type': 'noul',
   'noul': 0.8302,
   'confidence': 0.8302,
   'answer_confidence': 0.8302,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 190, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English Latin text',
  'detection': {'script': 'latin',
   'script_profile': {'latin': 1.0},
   'language': 'en',
   'is_english': True,
   'language_undecided': False,
   'd

## Laya Implementations of Use Cases

### Use Case 1: Customer Support Ticket Triage (Laya)
This use case demonstrates how to classify incoming customer support tickets. It assesses if an issue is an emergency, routes it to the appropriate department (e.g., Technical Support, Billing, Shipping), and estimates the customer's anger level, helping to prioritize and handle tickets efficiently.

In [11]:
from laya import Router

router = Router()

customer_ticket_state_1 = (
    "My internet has been down for 3 days! I can't work from home. This is an emergency! "
    "I need someone to fix this NOW. "
)

support_triage_questions_1 = {
    "is_emergency": {
        "type": "noul",
        "instructions": "Is the customer issue an active emergency?",
        "criteria": {
            "true": "The issue requires immediate, critical attention due to severe impact or danger.",
            "false": "The issue is standard and does not pose an immediate threat or severe disruption."
        }
    },
    "department_routing": {
        "type": "choice",
        "choices": ["Billing", "Technical Support", "Shipping"],
        "instructions": "Which department should handle this ticket?",
        "criteria": {
            "Billing": "Deals with invoices, payments, and account charges.",
            "Technical Support": "Handles product malfunctions, software issues, or technical guidance.",
            "Shipping": "Manages delivery, tracking, and logistics."
        }
    },
    "anger_level": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the customer's anger level from 1 (calm) to 5 (enraged)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_1 = router.predict(customer_ticket_state_1, support_triage_questions_1)

is_emergency_1 = result_1['answers']['is_emergency']['noul']
department_routing_1 = result_1['answers']['department_routing']['choice']
anger_level_1 = result_1['answers']['anger_level']['score']

print(f"Is Emergency: {is_emergency_1}")
print(f"Department Routing: {department_routing_1}")
print(f"Anger Level: {anger_level_1}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/router.py:456: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, **kwargs)


Is Emergency: 0.8716
Department Routing: Technical Support
Anger Level: 0.4388


In [12]:
result_1

{'model': 'laya-rl-agent',
 'answers': {'is_emergency': {'type': 'noul',
   'noul': 0.8716,
   'confidence': 0.8716,
   'answer_confidence': 0.8716,
   'action': {'act_probability': 1.0}},
  'department_routing': {'type': 'choice',
   'choice': 'Technical Support',
   'probabilities': {'Billing': 0.1166,
    'Technical Support': 0.6927,
    'Shipping': 0.1907},
   'confidence': 0.2528,
   'answer_confidence': 0.6927,
   'action': {'act_probability': 1.0}},
  'anger_level': {'type': 'score',
   'score': 0.4388,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.7158,
    '1': 0.1651,
    '2': 0.0904,
    '3': 0.0218,
    '4': 0.0068},
   'confidence': 0.4584,
   'answer_confidence': 0.7158,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 251, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English Latin text',
  'detection': {'script': 'latin',
   'script_profile': {'lati

### Use Case 2: Content Moderation and Brand Safety (Laya)
This use case focuses on moderating user-generated content for brand safety. It detects hate speech, categorizes the type of violation (e.g., harassment, profanity), and rates the overall toxicity level of the content, which is crucial for maintaining a safe online environment.

In [13]:
from laya import Router

router = Router()

user_comment_state_2 = "This website is terrible and anyone who uses it is an idiot. You should all go to hell!"

content_moderation_questions_2 = {
    "flag_hate_speech": {
        "type": "noul",
        "instructions": "Does this comment contain explicit hate speech?",
        "criteria": {
            "true": "The comment contains derogatory, discriminatory, or hateful language targeting a specific group or individual.",
            "false": "The comment does not contain explicit hate speech."
        }
    },
    "violation_type": {
        "type": "choice",
        "choices": ["spam", "harassment", "profanity", "hate_speech"],
        "instructions": "Categorize the primary violation type.",
        "criteria": {
            "spam": "Unsolicited or repetitive content.",
            "harassment": "Targeted abusive behavior.",
            "profanity": "Offensive language.",
            "hate_speech": "Expressions of hatred, discrimination, or violence against a protected group."
        }
    },
    "toxicity_level": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Rate the toxicity level from 1 (mild) to 10 (extreme)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

result_2 = router.predict(user_comment_state_2, content_moderation_questions_2)

flag_hate_speech_2 = result_2['answers']['flag_hate_speech']['noul']
violation_type_2 = result_2['answers']['violation_type']['choice']
toxicity_level_2 = result_2['answers']['toxicity_level']['score']

print(f"Flag Hate Speech: {flag_hate_speech_2}")
print(f"Violation Type: {violation_type_2}")
print(f"Toxicity Level: {toxicity_level_2}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Flag Hate Speech: 0.2183
Violation Type: harassment
Toxicity Level: 0.984


In [14]:
result_2

{'model': 'laya-rl-agent',
 'answers': {'flag_hate_speech': {'type': 'noul',
   'noul': 0.2183,
   'confidence': 0.7817,
   'answer_confidence': 0.7817,
   'action': {'act_probability': 1.0}},
  'violation_type': {'type': 'choice',
   'choice': 'harassment',
   'probabilities': {'spam': 0.1762,
    'harassment': 0.5622,
    'profanity': 0.208,
    'hate_speech': 0.0536},
   'confidence': 0.197,
   'answer_confidence': 0.5622,
   'action': {'act_probability': 1.0}},
  'toxicity_level': {'type': 'score',
   'score': 0.984,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0.8287,
    '1': 0.0399,
    '2': 0.0107,
    '3': 0.0086,
    '4': 0.0051,
    '5': 0.0052,
    '6': 0.0061,
    '7': 0.0099,
    '8': 0.0272,
    '9': 0.0585},
   'confidence': 0.666,
   'answer_confidence': 0.8287,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 246, 'out

### Use Case 3: Resume Screening and Candidate Pre-Qualification (Laya)
This use case helps in quickly pre-qualifying job applicants based on their resumes. It checks if mandatory experience requirements are met, determines the candidate's seniority tier (Junior, Mid, Senior, Lead), and rates their overall skill alignment for a specific role.

In [15]:
from laya import Router

router = Router()

resume_text_state_3 = (
    "John Doe. 10 years experience in Machine Learning. "
    "Led 3 major AI projects. PhD in Computer Science. "
    "Skills: Python, TensorFlow, PyTorch, AWS, Azure."
)

resume_screening_questions_3 = {
    "mandatory_experience_met": {
        "type": "noul",
        "instructions": "Does the applicant have at least 5 years of experience in Machine Learning?",
        "criteria": {
            "true": "Applicant explicitly states or implies 5 or more years of relevant experience.",
            "false": "Applicant states less than 5 years or does not provide enough information."
        }
    },
    "seniority_tier": {
        "type": "choice",
        "choices": ["Junior", "Mid", "Senior", "Lead"],
        "instructions": "Categorize the candidate's seniority tier.",
        "criteria": {
            "Junior": "Less than 3 years of experience.",
            "Mid": "3-5 years of experience.",
            "Senior": "5+ years of experience with leadership potential.",
            "Lead": "Extensive experience, proven leadership, and project management skills."
        }
    },
    "skill_alignment": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Rate the overall skill alignment with a typical Senior Machine Learning Engineer role (1-low, 10-high)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

result_3 = router.predict(resume_text_state_3, resume_screening_questions_3)

mandatory_experience_met_3 = result_3['answers']['mandatory_experience_met']['noul']
seniority_tier_3 = result_3['answers']['seniority_tier']['choice']
skill_alignment_3 = result_3['answers']['skill_alignment']['score']

print(f"Mandatory Experience Met: {mandatory_experience_met_3}")
print(f"Seniority Tier: {seniority_tier_3}")
print(f"Skill Alignment: {skill_alignment_3}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Mandatory Experience Met: 0.7308
Seniority Tier: Lead
Skill Alignment: 5.6602


In [16]:
result_3

{'model': 'laya-rl-agent',
 'answers': {'mandatory_experience_met': {'type': 'noul',
   'noul': 0.7308,
   'confidence': 0.7308,
   'answer_confidence': 0.7308,
   'action': {'act_probability': 1.0}},
  'seniority_tier': {'type': 'choice',
   'choice': 'Lead',
   'probabilities': {'Junior': 0.0431,
    'Mid': 0.1595,
    'Senior': 0.3498,
    'Lead': 0.4477},
   'confidence': 0.1665,
   'answer_confidence': 0.4477,
   'action': {'act_probability': 1.0}},
  'skill_alignment': {'type': 'score',
   'score': 5.6602,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0.0442,
    '1': 0.0695,
    '2': 0.0885,
    '3': 0.0988,
    '4': 0.045,
    '5': 0.057,
    '6': 0.0854,
    '7': 0.1225,
    '8': 0.2193,
    '9': 0.1698},
   'confidence': 0.0574,
   'answer_confidence': 0.2193,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 308, 'output_tokens

### Use Case 4: E-Commerce Fake Review Detection (Laya)
This use case aims to identify potentially fraudulent or incentivized product reviews on e-commerce platforms. It flags suspicious reviews, classifies the primary focus of the review (e.g., quality, price), and estimates its overall authenticity rating.

In [17]:
from laya import Router

router = Router()

product_review_state_4 = (
    "This product is absolutely amazing! I got it for free "
    "as part of a promotion. Best purchase ever!"
)

fake_review_questions_4 = {
    "flag_fraudulent_review": {
        "type": "noul",
        "instructions": "Is this review potentially fraudulent or incentivized?",
        "criteria": {
            "true": "The review mentions receiving the product for free, in exchange for a review, or shows other signs of unnatural positivity or repetition.",
            "false": "The review appears genuine and organic."
        }
    },
    "review_focus": {
        "type": "choice",
        "choices": ["quality", "price", "shipping", "customer_service"],
        "instructions": "Classify the primary focus of the review.",
        "criteria": {
            "quality": "Focuses on the product's durability, performance, or materials.",
            "price": "Comments on the cost, value for money, or deals.",
            "shipping": "Mentions delivery speed, packaging, or logistics.",
            "customer_service": "Refers to interactions with support or sales."
        }
    },
    "authenticity_rating": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Estimate the authenticity rating from 1 (very fake) to 5 (very authentic)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_4 = router.predict(product_review_state_4, fake_review_questions_4)

flag_fraudulent_review_4 = result_4['answers']['flag_fraudulent_review']['noul']
review_focus_4 = result_4['answers']['review_focus']['choice']
authenticity_rating_4 = result_4['answers']['authenticity_rating']['score']

print(f"Flag Fraudulent Review: {flag_fraudulent_review_4}")
print(f"Review Focus: {review_focus_4}")
print(f"Authenticity Rating: {authenticity_rating_4}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Flag Fraudulent Review: 0.181
Review Focus: quality
Authenticity Rating: 2.0093


In [18]:
result_4

{'model': 'laya-rl-agent',
 'answers': {'flag_fraudulent_review': {'type': 'noul',
   'noul': 0.181,
   'confidence': 0.819,
   'answer_confidence': 0.819,
   'action': {'act_probability': 1.0}},
  'review_focus': {'type': 'choice',
   'choice': 'quality',
   'probabilities': {'quality': 0.441,
    'price': 0.1291,
    'shipping': 0.2362,
    'customer_service': 0.1937},
   'confidence': 0.0737,
   'answer_confidence': 0.441,
   'action': {'act_probability': 1.0}},
  'authenticity_rating': {'type': 'score',
   'score': 2.0093,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.2655,
    '1': 0.1507,
    '2': 0.1432,
    '3': 0.1903,
    '4': 0.2503},
   'confidence': 0.0195,
   'answer_confidence': 0.2655,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 242, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English Latin text',
  'detection': {'script': 'latin',
   'script

### Use Case 5: Sentiment Analysis of Social Media Posts (Laya)
This use case analyzes the sentiment expressed in social media posts or comments. It determines if the sentiment is negative, categorizes the overall sentiment (positive, negative, neutral), and rates the intensity of that sentiment.

In [19]:
from laya import Router

router = Router()

social_media_post_state_5 = (
    "Just had the worst experience with my new phone. The battery dies so fast, "
    "and the camera is terrible. What a waste of money! #disappointed #techfail"
)

sentiment_analysis_questions_5 = {
    "is_negative_sentiment": {
        "type": "noul",
        "instructions": "Does the social media post express negative sentiment?",
        "criteria": {
            "true": "The post contains words, phrases, or overall tone indicating dissatisfaction, anger, or sadness.",
            "false": "The post is neutral or positive in sentiment."
        }
    },
    "sentiment_category": {
        "type": "choice",
        "choices": ["positive", "negative", "neutral"],
        "instructions": "Categorize the overall sentiment of the post.",
        "criteria": {
            "positive": "Expresses approval, happiness, or satisfaction.",
            "negative": "Expresses disapproval, sadness, or frustration.",
            "neutral": "No strong positive or negative emotion is conveyed."
        }
    },
    "sentiment_intensity": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the intensity of the sentiment from 1 (mild) to 5 (extreme)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_5 = router.predict(social_media_post_state_5, sentiment_analysis_questions_5)

is_negative_sentiment_5 = result_5['answers']['is_negative_sentiment']['noul']
sentiment_category_5 = result_5['answers']['sentiment_category']['choice']
sentiment_intensity_5 = result_5['answers']['sentiment_intensity']['score']

print(f"Is Negative Sentiment: {is_negative_sentiment_5}")
print(f"Sentiment Category: {sentiment_category_5}")
print(f"Sentiment Intensity: {sentiment_intensity_5}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Is Negative Sentiment: 0.8947
Sentiment Category: negative
Sentiment Intensity: 0.2956


In [20]:
result_5

{'model': 'laya-rl-agent',
 'answers': {'is_negative_sentiment': {'type': 'noul',
   'noul': 0.8947,
   'confidence': 0.8947,
   'answer_confidence': 0.8947,
   'action': {'act_probability': 1.0}},
  'sentiment_category': {'type': 'choice',
   'choice': 'negative',
   'probabilities': {'positive': 0.0122,
    'negative': 0.9725,
    'neutral': 0.0153},
   'confidence': 0.8683,
   'answer_confidence': 0.9725,
   'action': {'act_probability': 1.0}},
  'sentiment_intensity': {'type': 'score',
   'score': 0.2956,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.7333,
    '1': 0.2459,
    '2': 0.0153,
    '3': 0.003,
    '4': 0.0025},
   'confidence': 0.5844,
   'answer_confidence': 0.7333,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 258, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English Latin text',
  'detection': {'script': 'latin',
   'script_profile': {'latin'

### Use Case 6: Sales Lead Qualification and Intent Scoring (Laya)
This use case evaluates sales leads to determine their qualification and purchase intent. It identifies if a lead has an immediate budget, classifies their stage in the buyer's journey, and predicts their conversion potential.

In [21]:
from laya import Router

router = Router()

sales_lead_state_6 = (
    "Interested in your enterprise solution. Our budget is $100k, "
    "and we're looking to implement within the next 3 months. "
    "Currently evaluating vendors."
)

lead_qualification_questions_6 = {
    "has_immediate_budget": {
        "type": "noul",
        "instructions": "Does the lead have an immediate budget for the solution?",
        "criteria": {
            "true": "The lead explicitly states a budget or indicates readiness for financial commitment.",
            "false": "The lead does not mention a budget or indicates they are still early in the financial planning phase."
        }
    },
    "buyer_journey_stage": {
        "type": "choice",
        "choices": ["researching", "comparing", "ready_to_buy"],
        "instructions": "Classify the lead's buyer journey stage.",
        "criteria": {
            "researching": "Gathering information and understanding options.",
            "comparing": "Evaluating different solutions and vendors.",
            "ready_to_buy": "Prepared to make a purchase decision and looking for final quotes."
        }
    },
    "conversion_potential": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Grade the lead's conversion potential from 1 (low) to 10 (high)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

result_6 = router.predict(sales_lead_state_6, lead_qualification_questions_6)

has_immediate_budget_6 = result_6['answers']['has_immediate_budget']['noul']
buyer_journey_stage_6 = result_6['answers']['buyer_journey_stage']['choice']
conversion_potential_6 = result_6['answers']['conversion_potential']['score']

print(f"Has Immediate Budget: {has_immediate_budget_6}")
print(f"Buyer Journey Stage: {buyer_journey_stage_6}")
print(f"Conversion Potential: {conversion_potential_6}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Has Immediate Budget: 0.7159
Buyer Journey Stage: comparing
Conversion Potential: 6.0671


In [22]:
result_6

{'model': 'laya-rl-agent',
 'answers': {'has_immediate_budget': {'type': 'noul',
   'noul': 0.7159,
   'confidence': 0.7159,
   'answer_confidence': 0.7159,
   'action': {'act_probability': 1.0}},
  'buyer_journey_stage': {'type': 'choice',
   'choice': 'comparing',
   'probabilities': {'researching': 0.3063,
    'comparing': 0.3635,
    'ready_to_buy': 0.3302},
   'confidence': 0.0022,
   'answer_confidence': 0.3635,
   'action': {'act_probability': 1.0}},
  'conversion_potential': {'type': 'score',
   'score': 6.0671,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0.0216,
    '1': 0.0428,
    '2': 0.064,
    '3': 0.0913,
    '4': 0.0583,
    '5': 0.0734,
    '6': 0.1173,
    '7': 0.1307,
    '8': 0.2025,
    '9': 0.1983},
   'confidence': 0.0745,
   'answer_confidence': 0.2025,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 276, 'outp

### Use Case 7: News and Article Bias Guardrails (Laya)
This use case applies guardrails to news articles and similar content to detect potential biases. It flags misleading claims or unsubstantiated accusations, identifies political or commercial bias, and rates the factual objectivity of the text.

In [23]:
from laya import Router

router = Router()

news_article_state_7 = (
    "Breaking news: The government's new policy, widely condemned by experts, "
    "is clearly an attempt to suppress dissent and control information. "
    "This authoritarian move will undoubtedly lead to widespread protests."
)

bias_guardrails_questions_7 = {
    "has_misleading_claims": {
        "type": "noul",
        "instructions": "Does the text contain misleading claims or unsubstantiated accusations?",
        "criteria": {
            "true": "The text presents opinions as facts, uses loaded language, or makes claims without evidence.",
            "false": "The text is factual, balanced, and avoids deceptive language."
        }
    },
    "political_commercial_bias": {
        "type": "choice",
        "choices": ["liberal", "conservative", "neutral", "commercial_pro", "commercial_anti"],
        "instructions": "Identify the primary political or commercial bias.",
        "criteria": {
            "liberal": "Favors progressive policies and social change.",
            "conservative": "Favors traditional values and limited government.",
            "neutral": "Presents information objectively without a clear political or commercial leaning.",
            "commercial_pro": "Promotes a product, service, or company.",
            "commercial_anti": "Criticizes a product, service, or company."
        }
    },
    "factual_objectivity_rating": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the factual objectivity of the text from 1 (highly subjective/biased) to 5 (highly objective/unbiased)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_7 = router.predict(news_article_state_7, bias_guardrails_questions_7)

has_misleading_claims_7 = result_7['answers']['has_misleading_claims']['noul']
political_commercial_bias_7 = result_7['answers']['political_commercial_bias']['choice']
factual_objectivity_rating_7 = result_7['answers']['factual_objectivity_rating']['score']

print(f"Has Misleading Claims: {has_misleading_claims_7}")
print(f"Political/Commercial Bias: {political_commercial_bias_7}")
print(f"Factual Objectivity Rating: {factual_objectivity_rating_7}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Has Misleading Claims: 0.2304
Political/Commercial Bias: neutral
Factual Objectivity Rating: 0.5718


In [24]:
result_7

{'model': 'laya-rl-agent',
 'answers': {'has_misleading_claims': {'type': 'noul',
   'noul': 0.2304,
   'confidence': 0.7696,
   'answer_confidence': 0.7696,
   'action': {'act_probability': 1.0}},
  'political_commercial_bias': {'type': 'choice',
   'choice': 'neutral',
   'probabilities': {'liberal': 0.1437,
    'conservative': 0.2603,
    'neutral': 0.3371,
    'commercial_pro': 0.0601,
    'commercial_anti': 0.1988},
   'confidence': 0.0768,
   'answer_confidence': 0.3371,
   'action': {'act_probability': 1.0}},
  'factual_objectivity_rating': {'type': 'score',
   'score': 0.5718,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.5805,
    '1': 0.3107,
    '2': 0.0753,
    '3': 0.0235,
    '4': 0.01},
   'confidence': 0.3738,
   'answer_confidence': 0.5805,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 308, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English L

### Use Case 8: IT Security Incident Log Triage (Laya)
This use case assists in triaging IT security incidents from log data. It detects active intrusion attempts, classifies the type of attack (e.g., brute_force, SQL_injection), and assigns a severity level for immediate action by on-call engineers.

In [25]:
from laya import Router

router = Router()

security_log_state_8 = (
    "2024-03-08 10:35:12 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:15 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:18 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "Attack detected from 203.0.113.44 on port 80 - potential SQL injection attempt. Alert sent."
)

it_security_questions_8 = {
    "detect_intrusion_attempt": {
        "type": "noul",
        "instructions": "Does the log indicate an active intrusion attempt?",
        "criteria": {
            "true": "The log contains patterns indicative of unauthorized access, brute-force, or exploitation attempts.",
            "false": "The log shows routine activity or minor, non-malicious errors."
        }
    },
    "attack_type": {
        "type": "choice",
        "choices": ["DDoS", "brute_force", "SQL_injection", "malware_infection"],
        "instructions": "Classify the attack type.",
        "criteria": {
            "DDoS": "Distributed Denial of Service attack, overwhelming the system with traffic.",
            "brute_force": "Repeated, systematic attempts to guess credentials.",
            "SQL_injection": "Malicious code inserted into input fields to manipulate database queries.",
            "malware_infection": "System compromise by malicious software."
        }
    },
    "severity_level": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Prioritize the severity level for the on-call engineer from 1 (low) to 5 (critical)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_8 = router.predict(security_log_state_8, it_security_questions_8)

detect_intrusion_attempt_8 = result_8['answers']['detect_intrusion_attempt']['noul']
attack_type_8 = result_8['answers']['attack_type']['choice']
severity_level_8 = result_8['answers']['severity_level']['score']

print(f"Detect Intrusion Attempt: {detect_intrusion_attempt_8}")
print(f"Attack Type: {attack_type_8}")
print(f"Severity Level: {severity_level_8}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Detect Intrusion Attempt: 0.899
Attack Type: DDoS
Severity Level: 1.517


In [26]:
result_8

{'model': 'laya-rl-agent',
 'answers': {'detect_intrusion_attempt': {'type': 'noul',
   'noul': 0.899,
   'confidence': 0.899,
   'answer_confidence': 0.899,
   'action': {'act_probability': 1.0}},
  'attack_type': {'type': 'choice',
   'choice': 'DDoS',
   'probabilities': {'DDoS': 0.5373,
    'brute_force': 0.1163,
    'SQL_injection': 0.257,
    'malware_infection': 0.0893},
   'confidence': 0.1712,
   'answer_confidence': 0.5373,
   'action': {'act_probability': 1.0}},
  'severity_level': {'type': 'score',
   'score': 1.517,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.2573,
    '1': 0.4084,
    '2': 0.0912,
    '3': 0.0462,
    '4': 0.1969},
   'confidence': 0.1329,
   'answer_confidence': 0.4084,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 526, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English Latin text',
  'detection': {'script': 'latin',
   'scri

### Use Case 9: Medical Triage and Symptom Logging (Laya)
This use case helps in the initial triage of medical patients based on their intake notes. It flags critical life-threatening symptoms, recommends the appropriate medical specialty for referral, and ranks the urgency of care required.

In [27]:
from laya import Router

router = Router()

patient_notes_state_9 = (
    "Patient presents with severe chest pain radiating to the left arm, "
    "shortness of breath, and dizziness. Onset 30 minutes ago. No prior history."
)

medical_triage_questions_9 = {
    "flag_critical_symptoms": {
        "type": "noul",
        "instructions": "Does the patient exhibit critical life-threatening symptoms?",
        "criteria": {
            "true": "Symptoms indicate a severe medical emergency requiring immediate intervention (e.g., severe chest pain, difficulty breathing, sudden loss of consciousness).",
            "false": "Symptoms are non-life-threatening or mild."
        }
    },
    "medical_specialty": {
        "type": "choice",
        "choices": ["cardiology", "general_practice", "dermatology", "neurology", "emergency_medicine"],
        "instructions": "Recommend the appropriate medical specialty.",
        "criteria": {
            "cardiology": "Heart-related issues.",
            "general_practice": "Routine check-ups or common ailments.",
            "dermatology": "Skin conditions.",
            "neurology": "Brain and nervous system disorders.",
            "emergency_medicine": "Acute, life-threatening conditions."
        }
    },
    "urgency_of_care": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rank the urgency of care from 1 (routine) to 5 (immediate/critical)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_9 = router.predict(patient_notes_state_9, medical_triage_questions_9)

flag_critical_symptoms_9 = result_9['answers']['flag_critical_symptoms']['noul']
medical_specialty_9 = result_9['answers']['medical_specialty']['choice']
urgency_of_care_9 = result_9['answers']['urgency_of_care']['score']

print(f"Flag Critical Symptoms: {flag_critical_symptoms_9}")
print(f"Medical Specialty: {medical_specialty_9}")
print(f"Urgency of Care: {urgency_of_care_9}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Flag Critical Symptoms: 0.7856
Medical Specialty: emergency_medicine
Urgency of Care: 0.9712


In [28]:
result_9

{'model': 'laya-rl-agent',
 'answers': {'flag_critical_symptoms': {'type': 'noul',
   'noul': 0.7856,
   'confidence': 0.7856,
   'answer_confidence': 0.7856,
   'action': {'act_probability': 1.0}},
  'medical_specialty': {'type': 'choice',
   'choice': 'emergency_medicine',
   'probabilities': {'cardiology': 0.2288,
    'general_practice': 0.0849,
    'dermatology': 0.0259,
    'neurology': 0.0422,
    'emergency_medicine': 0.6183},
   'confidence': 0.3338,
   'answer_confidence': 0.6183,
   'action': {'act_probability': 1.0}},
  'urgency_of_care': {'type': 'score',
   'score': 0.9712,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.4212,
    '1': 0.3957,
    '2': 0.0685,
    '3': 0.0197,
    '4': 0.0949},
   'confidence': 0.2448,
   'answer_confidence': 0.4212,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 276, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'Engli

### Use Case 10: Contract and Legal Clause Risk Analysis (Laya)
This use case analyzes legal contracts to identify potential risks. It spots non-standard or unusual liability clauses, categorizes the legal domain of any notable clauses, and evaluates the overall financial exposure risk associated with the contract.

In [29]:
from laya import Router

router = Router()

legal_contract_state_10 = (
    "This agreement includes a clause for unlimited liability for the vendor, "
    "which is non-standard. Payment terms are net 90 days. "
    "Intellectual property rights are assigned exclusively to the client."
)

legal_risk_questions_10 = {
    "spot_non_standard_clauses": {
        "type": "noul",
        "instructions": "Does the contract contain non-standard or unusual liability clauses?",
        "criteria": {
            "true": "The contract includes terms that deviate significantly from industry norms regarding liability, indemnification, or warranty.",
            "false": "The contract's liability clauses are standard and acceptable."
        }
    },
    "legal_domain": {
        "type": "choice",
        "choices": ["indemnity", "payment_terms", "IP_rights", "termination_clauses"],
        "instructions": "Categorize the legal domain of the notable clause.",
        "criteria": {
            "indemnity": "Clauses related to protection against loss or damage.",
            "payment_terms": "Conditions for financial transactions and due dates.",
            "IP_rights": "Ownership and usage of intellectual property.",
            "termination_clauses": "Conditions under which the contract can be ended."
        }
    },
    "financial_exposure_risk": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Evaluate the overall financial exposure risk from 1 (low) to 5 (high)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_10 = router.predict(legal_contract_state_10, legal_risk_questions_10)

spot_non_standard_clauses_10 = result_10['answers']['spot_non_standard_clauses']['noul']
legal_domain_10 = result_10['answers']['legal_domain']['choice']
financial_exposure_risk_10 = result_10['answers']['financial_exposure_risk']['score']

print(f"Spot Non-Standard Clauses: {spot_non_standard_clauses_10}")
print(f"Legal Domain: {legal_domain_10}")
print(f"Financial Exposure Risk: {financial_exposure_risk_10}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Spot Non-Standard Clauses: 0.8679
Legal Domain: payment_terms
Financial Exposure Risk: 1.0557


In [30]:
result_10

{'model': 'laya-rl-agent',
 'answers': {'spot_non_standard_clauses': {'type': 'noul',
   'noul': 0.8679,
   'confidence': 0.8679,
   'answer_confidence': 0.8679,
   'action': {'act_probability': 1.0}},
  'legal_domain': {'type': 'choice',
   'choice': 'payment_terms',
   'probabilities': {'indemnity': 0.0296,
    'payment_terms': 0.8444,
    'IP_rights': 0.0904,
    'termination_clauses': 0.0356},
   'confidence': 0.5795,
   'answer_confidence': 0.8444,
   'action': {'act_probability': 1.0}},
  'financial_exposure_risk': {'type': 'score',
   'score': 1.0557,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.398,
    '1': 0.3168,
    '2': 0.1725,
    '3': 0.0568,
    '4': 0.0559},
   'confidence': 0.1562,
   'answer_confidence': 0.398,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 287, 'output_tokens': 0},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English Latin text',
  'detection': 

---